# Employee Performance Analysis — INX Future Inc.

### Business Problem

INX Future Inc. wants to understand the factors associated with employee performance and use historical employee data to predict performance ratings.

### Objective
Train and evaluate classification models that predict employee performance rating, identify the most important factors, and save the final model.

## 1. Import Libraries


In [1]:
# Data handling 
import pandas as pd 
import numpy as np 

# Train-test split and cross validation 
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate

# Data preprocessing 
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Machine learning pipeline
from sklearn.pipeline import Pipeline

# Clssification models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.model_selection import GridSearchCV

# Model evaluation 
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Visualization 
import matplotlib.pyplot as plt 
import seaborn as sns

# Save and load the trained model 
import joblib

## 2. Load the Cleaned Dataset

We load the cleaned employee dataset for further analysis and model training. We remove `EmpNumber` because it is only an employee identifier and doesn't help predict performance.

We also remove `Attrition` because it indicates whether an employee has left the company. This information isn't available when hiring a new candidate, so including it would make the model less suitable for that purpose.

In our dataset, employees who stayed had an average performance rating of 2.96, compared with 2.90 for employees who left. These averages are quite close, suggesting that Attrition has only a weak relationship with performance ratings in this dataset.

In [2]:
df = pd.read_csv("../../data/processed/employee_performance_processed.csv").drop(columns=["EmpNumber", "Attrition"])

In [3]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Dataset shape: (1200, 27)

Column names:
['Age', 'Gender', 'EducationBackground', 'MaritalStatus', 'EmpDepartment', 'EmpJobRole', 'BusinessTravelFrequency', 'DistanceFromHome', 'EmpEducationLevel', 'EmpEnvironmentSatisfaction', 'EmpHourlyRate', 'EmpJobInvolvement', 'EmpJobLevel', 'EmpJobSatisfaction', 'NumCompaniesWorked', 'OverTime', 'EmpLastSalaryHikePercent', 'EmpRelationshipSatisfaction', 'TotalWorkExperienceInYears', 'TrainingTimesLastYear', 'EmpWorkLifeBalance', 'ExperienceYearsAtThisCompany', 'ExperienceYearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'PerformanceRating', 'ExperienceBeforeCurrentCompany']


In [4]:
df.head()

,Age,Gender,EducationBackground,MaritalStatus,EmpDepartment,EmpJobRole,BusinessTravelFrequency,DistanceFromHome,EmpEducationLevel,EmpEnvironmentSatisfaction,...,EmpRelationshipSatisfaction,TotalWorkExperienceInYears,TrainingTimesLastYear,EmpWorkLifeBalance,ExperienceYearsAtThisCompany,ExperienceYearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,PerformanceRating,ExperienceBeforeCurrentCompany
0,32,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,10,3,4,...,4,10,2,2,10,7,0,8,3,0
1,47,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,14,4,4,...,4,20,2,3,7,7,1,7,3,13
2,40,Male,Life Sciences,Married,Sales,Sales Executive,Travel_Frequently,5,4,4,...,3,20,2,3,18,13,1,12,4,2
3,41,Male,Human Resources,Divorced,Human Resources,Manager,Travel_Rarely,10,4,2,...,2,23,2,2,21,6,12,6,3,2
4,60,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,16,4,1,...,4,10,1,3,2,2,2,2,3,8


# Part 1: With 8 Features

## 3. Define Features and Target for 8 features

In [5]:
target = "PerformanceRating"

selected_features = [
    "EducationBackground",
    "EmpEducationLevel",
    "EmpDepartment",
    "EmpJobRole",
    "EmpJobLevel",
    "NumCompaniesWorked",
    "TotalWorkExperienceInYears",
    "BusinessTravelFrequency"
] 

X = df[selected_features]
y = df[target]

In [6]:
print("Input features shape:", X.shape)
print("Target shape:", y.shape)

Input features shape: (1200, 8)
Target shape: (1200,)


## 4. Train-Test Split for 8 features

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify= y , random_state=42)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)

print("\nTraining target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting target distribution:")
print(y_test.value_counts().sort_index())

Training features shape: (960, 8)
Testing features shape: (240, 8)

Training target shape: (960,)
Testing target shape: (240,)

Training target distribution:
PerformanceRating
2    155
3    699
4    106
Name: count, dtype: int64

Testing target distribution:
PerformanceRating
2     39
3    175
4     26
Name: count, dtype: int64


## 5. Preprocessing Function

In [8]:
def preprocessor(X): 
    numerical_features = X.select_dtypes(include="number").columns.tolist() 

    categorical_features = X.select_dtypes(include="object").columns.tolist()

    numerical_pipeline = Pipeline(steps=[
        ("scaler", StandardScaler())
    ])
    categorical_pipeline = Pipeline(steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    column_transformer = ColumnTransformer(transformers=[
    ("num_pipeline", numerical_pipeline, numerical_features),
    ("cat_pipeline", categorical_pipeline, categorical_features)
    ])

    return column_transformer

In [9]:
preprocessor_8 = preprocessor(X)

## 6. Classification Models Building for 8 features

### 7.1 Logistic Regression

In [10]:
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor_8), 
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

### 7.2 Decision Tree

In [11]:
decision_model = Pipeline(steps=[
    ("preprocessor", preprocessor_8), 
    ("classifier", DecisionTreeClassifier(random_state=42))
])

### 7.3 Random Forest

In [12]:
randomforest_model = Pipeline(steps=[
    ("preprocessor", preprocessor_8),
    ("classifier", RandomForestClassifier(random_state=42))
])

### 7.4 Gradient Boosting

In [13]:
gradientboost_model = Pipeline(steps=[
    ("preprocessor", preprocessor_8), 
    ("classifier", GradientBoostingClassifier(random_state=42))
])

## 7. Model Training Using 8 Features

In this experiment, I trained four classification models using eight selected
features to predict employee performance ratings.

I used Logistic Regression, Decision Tree, Random Forest, and Gradient
Boosting. The purpose was to establish a baseline and understand how well
these models perform using a smaller set of features.

All models were trained using the same training dataset so their performance
could be compared fairly.

In [14]:
# Train logistic regressor
logistic_model.fit(X_train, y_train)

# Train Decision Tree 
decision_model.fit(X_train, y_train)

# Train Random Forest 
randomforest_model.fit(X_train, y_train)

# Train Gradient Boosting 
gradientboost_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num_pipeline', ...), ('cat_pipeline', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 8. Model Evaluation Using Selected Features

After training the models, I evaluated their performance on the test dataset.

I used accuracy, macro precision, macro recall, and macro F1-score to
understand their performance from different perspectives.

Macro metrics are particularly useful here because the number of employees
in each performance rating class is not equal.

In [15]:
# Define the evaluation function 
def evaluate_model(model, X_test, y_test): 
    y_pred = model.predict(X_test)

    return {
        "Accuracy":accuracy_score(y_test, y_pred),
        "Precision_Macro":precision_score(y_test, y_pred, average="macro", zero_division=0), 
        "Recall_Macro": recall_score(y_test, y_pred, average="macro", zero_division=0), 
        "F1_Macro": f1_score(y_test, y_pred, average="macro", zero_division=0)
    }

In [16]:
# Evaluate the four models 

models = {
    "Logistic Regression":logistic_model, 
    "Decision Tree":decision_model, 
    "Random Forest":randomforest_model, 
    "Gradient Boosting":gradientboost_model
}

results = []

for model_name, model in models.items(): 
    metrics = evaluate_model(model, X_test, y_test)
    metrics["Model"] = model_name
    results.append(metrics)

model_comparison = pd.DataFrame(results)

model_comparison = model_comparison[
    ["Model", "Accuracy", "Precision_Macro", "Recall_Macro", "F1_Macro"]
]

model_comparison = model_comparison.sort_values(by="F1_Macro", ascending=False).reset_index(drop=True)

model_comparison.round(4)

,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Decision Tree,0.5792,0.3514,0.3440,0.3459
1,Random Forest,0.7042,0.2805,0.3285,0.2904
2,Logistic Regression,0.7250,0.2427,0.3314,0.2802
3,Gradient Boosting,0.7083,0.2422,0.3238,0.2771


In [17]:
# Clssification Report 
for model_name, model in models.items():
    print(f"\n{'=' * 60}")
    print(model_name)
    print("=" * 60)

    y_pred = model.predict(X_test)

    print(
        classification_report(
            y_test,
            y_pred,
            labels=[2, 3, 4],
            zero_division=0
        )
    )


Logistic Regression
              precision    recall  f1-score   support

           2       0.00      0.00      0.00        39
           3       0.73      0.99      0.84       175
           4       0.00      0.00      0.00        26

    accuracy                           0.72       240
   macro avg       0.24      0.33      0.28       240
weighted avg       0.53      0.72      0.61       240


Decision Tree
              precision    recall  f1-score   support

           2       0.24      0.18      0.21        39
           3       0.71      0.74      0.72       175
           4       0.10      0.12      0.11        26

    accuracy                           0.58       240
   macro avg       0.35      0.34      0.35       240
weighted avg       0.57      0.58      0.57       240


Random Forest
              precision    recall  f1-score   support

           2       0.11      0.03      0.04        39
           3       0.73      0.96      0.83       175
           4       0.00 

### Finding: 8-Feature Model Evaluation

The results show that the models struggled to predict all three performance ratings correctly. Accuracy alone doesn't give us the full picture because most employees in the test dataset have a rating of 3. A model can achieve reasonable accuracy by predicting the majority class while still failing to identify employees with ratings 2 and 4. Logistic Regression failed to correctly identify either of these two groups, and the Decision Tree performed slightly better, with a macro F1-score of 0.346, but its overall performance was still weak.

These eight features mainly describe an employee's background and job details, such as education, department, job role, work experience, and business travel. They don't include factors such as work environment satisfaction, salary hike percentage, and years since the last promotion, which were among the most important factors found during our analysis. This helps explain why the models using all available predictor features performed better.

These eight features are also closer to the information available when hiring a new employee. The results suggest that background information alone may not be enough to predict future performance reliably. This is an important limitation to consider if the model is used to support hiring decisions.

# Part 2: With all Features

## 9. Define all features dataset.

In [18]:
X_all = df.drop(columns=["PerformanceRating"])
y_all = df["PerformanceRating"]

## 10. Split all features data

In [19]:
X_all_train, X_all_test, y_all_train, y_all_test = train_test_split(X_all, y_all, test_size=0.20, stratify=y_all , random_state=42)

## 11. Models building for all features

In [20]:
preprocessor_all = preprocessor(X_all)

In [21]:
# Logistic regression 
logistic_model_all = Pipeline(steps=[
    ("preprocessor", preprocessor_all), 
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

# Decision Tree
decision_model_all = Pipeline(steps=[
    ("preprocessor", preprocessor_all),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

# Random forest 
randomforest_model_all = Pipeline(steps=[
    ("preprocessor", preprocessor_all), 
    ("classifier", RandomForestClassifier(random_state=42))
])

# Gradient boosting 
gradientboost_model_all = Pipeline(steps=[
    ("preprocessor", preprocessor_all), 
    ("classifier", GradientBoostingClassifier(random_state=42))
])

## 12. Model Training Using All Available Predictor Features

In [22]:
# Train logistic regression using all features 
logistic_model_all.fit(X_all_train, y_all_train)

# Train decision tree using all features 
decision_model_all.fit(X_all_train, y_all_train)

# Train random forest using all features 
randomforest_model_all.fit(X_all_train, y_all_train)

# Train gradient boosting using all features 
gradientboost_model_all.fit(X_all_train, y_all_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num_pipeline', ...), ('cat_pipeline', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 13. Model Evaluation Using All Features

After training the four classification models using all 26 predictor features,
I evaluated their performance on the test dataset.

I used accuracy, macro precision, macro recall, and macro F1-score to compare
the models and understand how well they predicted the three employee
performance rating classes.

These metrics help evaluate both overall prediction accuracy and the
models' ability to identify different performance rating classes.

In [23]:
# Evaluate the four models 

all_models = {
    "Logistic Regression":logistic_model_all, 
    "Decision Tree":decision_model_all, 
    "Random Forest":randomforest_model_all, 
    "Gradient Boosting":gradientboost_model_all
}

results_all = []

for model_name, model in all_models.items(): 
    metrics = evaluate_model(model, X_all_test, y_all_test)
    metrics["Model"] = model_name
    results_all.append(metrics)

model_comparison = pd.DataFrame(results_all)

model_comparison = model_comparison[
    ["Model", "Accuracy", "Precision_Macro", "Recall_Macro", "F1_Macro"]
]

model_comparison = model_comparison.sort_values(by="F1_Macro", ascending=False).reset_index(drop=True)

model_comparison.round(4)

,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro
0,Gradient Boosting,0.9292,0.9134,0.8623,0.8856
1,Decision Tree,0.8917,0.8451,0.8361,0.8405
2,Random Forest,0.9000,0.9304,0.7763,0.8360
3,Logistic Regression,0.8208,0.7643,0.6761,0.7121


# Part 3: Model Comparison and Finalization

## 14. Compare 8 features and All features Results

In [24]:
baseline_results = pd.DataFrame(results)

all_results = pd.DataFrame(results_all)

In [25]:
print("Baseline Model:\n", baseline_results)
print("\nAll feature Model:\n", all_results)

Baseline Model:
    Accuracy  Precision_Macro  Recall_Macro  F1_Macro                Model
0  0.725000         0.242678      0.331429  0.280193  Logistic Regression
1  0.579167         0.351362      0.344005  0.345915        Decision Tree
2  0.704167         0.280515      0.328547  0.290432        Random Forest
3  0.708333         0.242165      0.323810  0.277099    Gradient Boosting

All feature Model:
    Accuracy  Precision_Macro  Recall_Macro  F1_Macro                Model
0  0.820833         0.764333      0.676093  0.712144  Logistic Regression
1  0.891667         0.845122      0.836117  0.840512        Decision Tree
2  0.900000         0.930397      0.776337  0.836030        Random Forest
3  0.929167         0.913403      0.862271  0.885604    Gradient Boosting


### Findings and Interpretation

The baseline models were trained using a smaller set of selected features, while the all-feature models used all 26 input features. The results show that all four models performed better when all features were included.

Logistic Regression improved its accuracy from 72.50% to 82.08%. Decision Tree improved from 57.92% to 89.17%, while Random Forest increased from 70.42% to 90.00%. Gradient Boosting achieved the highest accuracy, increasing from 70.83% to 92.92%.

Among the all-feature models, Gradient Boosting also achieved the highest Macro F1-score of 0.8856, followed by Decision Tree at 0.8405 and Random Forest at 0.8360. This shows that Gradient Boosting performed best overall among the four models tested.

Based on these results, Gradient Boosting was selected for hyperparameter tuning to check whether its performance could be improved further.

## 15. Hyperparameter Tuning of Gradient Boosting

In [26]:
gb_tuning_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_all), 
    ("classifier", GradientBoostingClassifier(random_state=42))
])

In [27]:
param_grid = {
    "classifier__n_estimators": [100, 150, 300], 
    "classifier__learning_rate": [0.05, 0.1], 
    "classifier__max_depth": [2, 3, 4], 
    "classifier__min_samples_split": [2, 5, 7], 
    "classifier__min_samples_leaf": [1, 2, 4]
}

In [28]:
grid_search = GridSearchCV(
    estimator= gb_tuning_pipeline, 
    param_grid= param_grid, 
    scoring= "f1_macro", 
    cv= 5, 
    n_jobs=-1, 
    verbose=1
)

In [29]:
grid_search.fit(X_all_train, y_all_train)

Fitting 5 folds for each of 162 candidates, totalling 810 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__learning_rate': [0.05, 0.1], 'classifier__max_depth': [2, 3, ...], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 5, ...], ...}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num_pipeline', ...), ('cat_pipeline', ...)]"


In [30]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation Macro F1 Score:")
print(round(grid_search.best_score_, 4))

Best Parameters:
{'classifier__learning_rate': 0.1, 'classifier__max_depth': 4, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 300}

Best Cross-Validation Macro F1 Score:
0.8975


In [31]:
# Retrieve the best tuned pipeline
best_gb_model = grid_search.best_estimator_

print("Best tuned model retrieved successfully.")

Best tuned model retrieved successfully.


In [32]:
# Predict employee performance ratings
y_pred_tuned = best_gb_model.predict(X_all_test)

In [33]:
# Evaluate the tuned Gradient Boosting model

tuned_accuracy = accuracy_score(y_all_test, y_pred_tuned)

tuned_precision = precision_score(
    y_all_test,
    y_pred_tuned,
    average="macro",
    zero_division=0
)

tuned_recall = recall_score(
    y_all_test,
    y_pred_tuned,
    average="macro",
    zero_division=0
)

tuned_f1 = f1_score(
    y_all_test,
    y_pred_tuned,
    average="macro",
    zero_division=0
)

print("Tuned Gradient Boosting Model Results")
print("-" * 40)
print(f"Accuracy:        {tuned_accuracy:.4f}")
print(f"Macro Precision: {tuned_precision:.4f}")
print(f"Macro Recall:    {tuned_recall:.4f}")
print(f"Macro F1 Score:  {tuned_f1:.4f}")

Tuned Gradient Boosting Model Results
----------------------------------------
Accuracy:        0.9292
Macro Precision: 0.9252
Macro Recall:    0.8556
Macro F1 Score:  0.8866


### Finding: Hyperparameter Tuning

Hyperparameter tuning resulted in a small improvement in the macro F1-score, from 0.8856 to 0.8866, while accuracy remained at 92.92%. This suggests that the original model was already performing well, and tuning provided only a slight improvement. Some of the best parameter values were at the boundaries of the search ranges, so testing a wider range could be considered in future work.

## 16. Analyze Feature Importance

In [34]:
best_model = best_gb_model

preprocessor_fitted = best_model.named_steps["preprocessor"]
classifier = best_model.named_steps["classifier"]

feature_names = preprocessor_fitted.get_feature_names_out()
importance_values = classifier.feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
})

feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

feature_importance_df.head(10)

,Feature,Importance
0,num_pipeline__EmpEnvironmentSatisfaction,0.260412
1,num_pipeline__EmpLastSalaryHikePercent,0.240814
2,num_pipeline__YearsSinceLastPromotion,0.190149
3,cat_pipeline__EmpDepartment_Development,0.078364
4,num_pipeline__ExperienceYearsInCurrentRole,0.048788
5,num_pipeline__EmpWorkLifeBalance,0.048726
6,num_pipeline__YearsWithCurrManager,0.019911
7,cat_pipeline__EmpJobRole_Developer,0.015168
8,num_pipeline__ExperienceYearsAtThisCompany,0.013567
9,num_pipeline__DistanceFromHome,0.010632


### Finding: Feature Importance

The analysis shows that employee environment satisfaction, salary hike percentage, and years since the last promotion are the three most important features in the tuned Gradient Boosting model. Together, they account for about 69% of the model's impurity-based feature importance.

The department and job role also contribute to the predictions. For example, `EmpDepartment_Development` represents employees in the Development department, rather than a separate feature collected independently.

These results show which features the model relies on when making predictions, but they don't prove that these factors directly cause better performance. Salary hikes may also reflect employees' previous performance ratings, so this relationship should be interpreted carefully.

## 17. Evaluate the Final Model Using a Classification Report and Confusion Matrix

In [35]:
print("Classification Report:")
print(classification_report(y_all_test, y_pred_tuned, labels=[2, 3, 4], target_names=["Rating 2", "Rating 3", "Rating 4"], zero_division=0))

Classification Report:
              precision    recall  f1-score   support

    Rating 2       0.89      0.82      0.85        39
    Rating 3       0.93      0.98      0.96       175
    Rating 4       0.95      0.77      0.85        26

    accuracy                           0.93       240
   macro avg       0.93      0.86      0.89       240
weighted avg       0.93      0.93      0.93       240



In [36]:
cm = confusion_matrix(y_all_test, y_pred_tuned, labels=[2, 3, 4])

cm_df = pd.DataFrame(
    cm, 
    index=["Actual Rating 2", "Actual Rating 3", "Actual Rating 4"], 
    columns=["Predicted Rating 2", "Predicted Rating 3", "Predicted Rating 4"]
)
display(cm_df)

,Predicted Rating 2,Predicted Rating 3,Predicted Rating 4
Actual Rating 2,32,6,1
Actual Rating 3,4,171,0
Actual Rating 4,0,6,20


### Finding and Interpretation 

The tuned Gradient Boosting model was evaluated using a classification report and confusion matrix to understand how well it predicts the three employee performance ratings. The model achieved 92.92% accuracy, correctly predicting 223 out of 240 employees in the test dataset.

The classification report shows that the model performed best for Rating 3, with an F1-score of 0.96 and a recall of 0.98. For Rating 2, it achieved an F1-score of 0.85, correctly identifying 32 out of 39 employees. For Rating 4, it also achieved an F1-score of 0.85, correctly identifying 20 out of 26 employees.

The confusion matrix shows that most of the incorrect predictions happened when the model confused Rating 2 or Rating 4 with Rating 3. In particular, 6 employees with an actual Rating 4 were predicted as Rating 3. This suggests that the model has some difficulty distinguishing between these two performance categories.

Overall, the model performed well on the test data, especially when predicting Rating 3. However, there is still room to improve its predictions for Rating 2 and Rating 4. These results help us understand the model's strengths and limitations and identify areas for further improvement.

The model can be used as a starting point for analyzing employee performance, but its predictions should be treated as supporting information rather than the sole basis for employee-related decisions.

## 18. Saving the Final Model

After evaluating the tuned Gradient Boosting model, the next step is to save it for future use. Saving the complete pipeline allows us to reuse the same preprocessing steps and trained model without repeating the training process.

The saved model can then be loaded into `predict_model.ipynb` to predict employee performance ratings for new input data.

In [37]:
from pathlib import Path
model_path = Path("gradient_boosting_final.pkl")

joblib.dump(best_gb_model, model_path)

['gradient_boosting_final.pkl']

### Model Input Features

The trained model expects 26 input features for making predictions. These include employee demographics, education, job details, satisfaction scores, work experience, and other employment-related information.

The input data must contain the following columns in the expected format:

1. Age
2. Gender
3. EducationBackground
4. MaritalStatus
5. EmpDepartment
6. EmpJobRole
7. BusinessTravelFrequency
8. DistanceFromHome
9. EmpEducationLevel
10. EmpEnvironmentSatisfaction
11. EmpHourlyRate
12. EmpJobInvolvement
13. EmpJobLevel
14. EmpJobSatisfaction
15. NumCompaniesWorked
16. OverTime
17. EmpLastSalaryHikePercent
18. EmpRelationshipSatisfaction
19. TotalWorkExperienceInYears
20. TrainingTimesLastYear
21. EmpWorkLifeBalance
22. ExperienceYearsAtThisCompany
23. ExperienceYearsInCurrentRole
24. YearsSinceLastPromotion
25. YearsWithCurrManager
26. ExperienceBeforeCurrentCompany

The columns `EmpNumber`, `Attrition`, and `PerformanceRating` are excluded from the input features. `PerformanceRating` is the target the model predicts, while `ExperienceBeforeCurrentCompany` is an engineered feature calculated during data preparation.

## 19. Limitation for Hiring Use

The model using all available features performs well when predicting ratings for existing employees. However, some of its most important features, such as environment satisfaction, salary hike percentage, promotion history, and company tenure, are only available after someone joins the company.

The experiment using eight background and job-related features performed much less effectively. This shows that predicting employee performance using information available at hiring is more difficult.

Therefore, the current model is better suited to analysing performance patterns among existing employees than predicting the performance of new candidates. It should only be used as a supporting analytical tool, not as the sole basis for hiring decisions.

## 20. Conclusion

In this project, we analysed employee data, explored the factors related to performance ratings, and trained several classification models. We compared models using eight features with models using all available predictor features.

The Gradient Boosting model performed best among the models evaluated using all available features, achieving 92.92% accuracy and a macro F1-score of approximately 0.89 on the test dataset.

Our analysis highlighted employee environment satisfaction, salary hike percentage, and years since the last promotion as the most important features in the final model. These findings show patterns in the available data, but they do not prove that these factors directly cause better performance.

The results also showed that predicting performance using only background and job-related information was more difficult. Therefore, the model is more suitable for analysing existing employees than making predictions about new candidates.